# RQ2 Evaluation Suite: Metadata Extraction Accuracy

This notebook implements the complete, peer-defensible evaluation for **Research Question 2 (RQ2)** and **Hypothesis H2**.

### Revised RQ2 & H2 Definitions:
*   **RQ2:** *How accurately can the LLM agent extract bibliographic metadata (title, year, venue, and authors) compared with externally curated bibliographic records, and semantic attributes (topics and concepts) compared with expert-authored annotations?*
*   **H2:** *The LLM agent is expected to achieve higher alignment for bibliographic metadata, measured using exact match and author-set F1, than for semantic attributes, measured using set-based Precision, Recall, and F1 against expert-authored annotations.*

### Evaluation Architecture:
1.  **Group A (Bibliographic Metadata - 40 papers):** Evaluated against externally curated records from **Semantic Scholar** (with OpenAlex fallback, using retries and incremental caching).
2.  **Group B1 (Strict Expert Vault Semantic Match - 22 papers):** Set-based Precision, Recall, and F1 comparing the agent's `hasTopic only` to Emile's vault `topics`.
2.  **Group B1 (Strict Expert Vault Semantic Match - 22 papers):** Set-based Precision, Recall, and F1 comparing the agent's `hasTopic` to Emile's vault `topics` (paper-title links removed).


In [19]:
import json
import re
import os
import sys
import time
from pathlib import Path
import yaml
import requests

print("Python Environment Setup Complete.")

Python Environment Setup Complete.


In [20]:
# ===========================================================================
# 1. Fail-Fast Integrity Checks & .env Loader
# ===========================================================================

# Read .env file manually if it exists to populate os.environ
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip("'").strip('"')

S2_API_KEY = os.environ.get("S2_API_KEY", "").strip()
OPENALEX_KEY = os.environ.get("OPENALEX_API_KEY", "").strip()

if not S2_API_KEY:
    raise ValueError("CRITICAL ERROR: S2_API_KEY environment variable is empty or not set!")
if not OPENALEX_KEY:
    raise ValueError("CRITICAL ERROR: OPENALEX_API_KEY environment variable is empty or not set!")

notes_dir = Path("data/generated_notes")
gold_metadata_path = Path("data/gold_labels/emile_vault_gold.json")
vault_arxiv_path = Path("data/gold_labels/emile_vault_arxiv_ids.json")

# Assertions
assert notes_dir.exists(), f"Directory not found: {notes_dir}"
note_files = sorted(list(notes_dir.glob("*.md")))
assert len(note_files) == 40, f"Expected exactly 40 generated notes, but found {len(note_files)} in {notes_dir}"

assert gold_metadata_path.exists(), f"Gold metadata file missing: {gold_metadata_path}"
assert vault_arxiv_path.exists(), f"Vault arXiv IDs map missing: {vault_arxiv_path}"

print("Integrity checks passed successfully!")
print(f"  - Found exactly {len(note_files)} generated notes in {notes_dir}.")
print("  - All required gold standard metadata files are present.")
print("  - API keys are verified in the environment.")

Integrity checks passed successfully!
  - Found exactly 40 generated notes in data\generated_notes.
  - All required gold standard metadata files are present.
  - API keys are verified in the environment.


In [21]:
# ===========================================================================
# 2. Helper Functions (Normalization & Venue Matching)
# ===========================================================================

def normalise(s: str) -> str:
    """Lowercase, strip punctuation and collapse whitespace for fuzzy comparison."""
    if not isinstance(s, str):
        s = str(s or "")
    # Resolve WikiLinks to use the canonical target (the part before |)
    s = re.sub(r"\[\[([^\]]+)\]\]", lambda m: m.group(1).split('|')[0], s)
    return re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()

def normalise_venue(s: str) -> str:
    """General normalisation followed by year and digit stripping for venues."""
    s = normalise(s)
    # Remove all digits (years, conference edition numbers like 19th, 25)
    s = re.sub(r"\d+", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Collapse all arxiv/preprint variants to a single token 'arxiv'
    if s in ["arxiv", "arxivorg", "arxiv org", "arxivpreprint", "arxiv preprint", "preprint"]:
        return "arxiv"
    return s

def extract_initials(title: str) -> str:
    """Extract initials from a venue title, removing only connector words."""
    connector_words = {"of", "on", "the", "and", "in", "for", "with", "to", "at", "by", "an", "a"}
    words = [w for w in normalise_venue(title).split() if w not in connector_words]
    return "".join(w[0] for w in words if w)

def fuzzy_venue_match(pred: str, gold: str, threshold: float = 0.70) -> bool:
    """
    Evaluates venue match using normalized string distance, acronym matching,
    known abbreviations dictionary, and Jaccard token similarity.
    """
    p_norm = normalise_venue(pred or "")
    g_norm = normalise_venue(gold or "")
    
    if not p_norm or not g_norm:
        return False
    
    # 1. Direct match
    if p_norm == g_norm:
        return True
        
    # 2. Known Conference Synonym Mapping
    VENUE_SYNONYMS = {
        "neurips": ["neural information processing systems", "advances in neural information processing systems", "nips"],
        "aaai": ["association for the advancement of artificial intelligence", "aaai conference on artificial intelligence"],
        "uai": ["conference on uncertainty in artificial intelligence", "uncertainty in artificial intelligence"],
        "icml": ["international conference on machine learning"],
        "iclr": ["international conference on learning representations", "international conference on representations"],
        "aistats": ["international conference on artificial intelligence and statistics"],
        "www": ["the web conference", "web conference", "www acm web conference", "acm web conference", "www the acm web conference"]
    }
    
    for sync_key, names in VENUE_SYNONYMS.items():
        pred_match = (p_norm == sync_key or any(normalise_venue(n) == p_norm for n in names))
        gold_match = (g_norm == sync_key or any(normalise_venue(n) == g_norm for n in names))
        if pred_match and gold_match:
            return True
            
    # 3. Acronym matching check (e.g. 'ICML' vs 'International Conference on Machine Learning')
    p_initials = extract_initials(pred)
    g_initials = extract_initials(gold)
    if p_norm == g_initials or g_norm == p_initials or (p_initials == g_initials and len(p_initials) > 1):
        return True

    # 4. Jaccard Token similarity (excluding connectors)
    connector_words = {"of", "on", "the", "and", "in", "for", "with", "to", "at", "by", "an", "a"}
    p_tokens = {t for t in p_norm.split() if t not in connector_words}
    g_tokens = {t for t in g_norm.split() if t not in connector_words}
    
    if not p_tokens or not g_tokens:
        return False
        
    intersection = len(p_tokens.intersection(g_tokens))
    union = len(p_tokens.union(g_tokens))
    jaccard = intersection / union if union > 0 else 0.0
    
    return jaccard >= threshold

def compute_set_prf(pred_list: list[str], gold_list: list[str]) -> dict:
    """Compute set-based Precision, Recall, and F1 with normalized values."""
    pred_set = {normalise(x) for x in (pred_list or []) if normalise(x)}
    gold_set = {normalise(x) for x in (gold_list or []) if normalise(x)}
    
    if not gold_set:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0, "tp": 0, "n_gold": 0, "n_pred": len(pred_set)}
        
    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "n_gold": len(gold_set),
        "n_pred": len(pred_set)
    }


In [22]:
# ===========================================================================
# 3. API Fetching Modules (S2 & OpenAlex with retries)
# ===========================================================================

S2_HEADERS = {"x-api-key": S2_API_KEY}

def fetch_semantic_scholar_record(arxiv_id: str, retries: int = 3) -> dict | None:
    """Queries Semantic Scholar for paper bibliographic metadata, with backoff retries."""
    url = f"https://api.semanticscholar.org/graph/v1/paper/arXiv:{arxiv_id}?fields=title,authors,year,venue"
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=S2_HEADERS, timeout=10)
            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                sleep_time = 5 * (attempt + 1)
                print(f"  [S2 Rate Limit] 429 for {arxiv_id}. Sleeping {sleep_time}s (Attempt {attempt+1}/{retries})...")
                time.sleep(sleep_time)
            else:
                return None
        except Exception as e:
            print(f"  [S2 Exception] {arxiv_id}: {e}")
            time.sleep(2)
    return None

def fetch_openalex_record(arxiv_id: str, retries: int = 3) -> dict | None:
    """Queries OpenAlex for a paper work using its capitalized arXiv DOI filter."""
    doi_url = f"https://doi.org/10.48550/arXiv.{arxiv_id}"
    url = f"https://api.openalex.org/works?filter=doi:{doi_url}&api_key={OPENALEX_KEY}"
    headers = {"User-Agent": "mailto:jubam@student.vu.nl"}
    
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                results = response.json().get("results", [])
                return results[0] if results else None
            elif response.status_code == 429:
                sleep_time = 5 * (attempt + 1)
                print(f"  [OpenAlex Rate Limit] 429 for {arxiv_id}. Sleeping {sleep_time}s (Attempt {attempt+1}/{retries})...")
                time.sleep(sleep_time)
            else:
                return None
        except Exception as e:
            print(f"  [OpenAlex Exception] {arxiv_id}: {e}")
            time.sleep(2)
    return None

In [23]:
# ===========================================================================
# 4. Group A — Bibliographic Metadata Evaluation (with incremental caching)
# ===========================================================================

def evaluate_group_a(notes_dir: Path, output_gold_path: Path) -> dict:
    note_files = sorted(list(notes_dir.glob("*.md")))
    all_arxiv_ids = [nf.stem for nf in note_files]
    
    # Incremental Cache Loading
    gold_standard = {}
    if output_gold_path.exists():
        with open(output_gold_path, encoding="utf-8") as f:
            try:
                gold_standard = json.load(f) or {}
            except Exception:
                gold_standard = {}
                
    missing_ids = [aid for aid in all_arxiv_ids if aid not in gold_standard]
    
    if missing_ids:
        print(f"Group A: Found {len(gold_standard)} cached records. Fetching {len(missing_ids)} missing records...")
        cache_updated = False
        for idx, aid in enumerate(missing_ids, 1):
            print(f"  [{idx}/{len(missing_ids)}] Fetching S2 metadata for {aid}...")
            record = fetch_semantic_scholar_record(aid)
            if record:
                gold_standard[aid] = {
                    "title": record.get("title", ""),
                    "year": record.get("year"),
                    "venue": record.get("venue", ""),
                    "authors": [a["name"] for a in record.get("authors", []) if "name" in a]
                }
                cache_updated = True
            time.sleep(1.2)  # rate limit spacing
            
        if cache_updated:
            output_gold_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_gold_path, "w", encoding="utf-8") as f:
                json.dump(gold_standard, f, indent=2, ensure_ascii=False)
    else:
        print(f"Group A: Cache is complete. Loaded {len(gold_standard)} records.")
        
    unmatched_papers = []
    group_a_details = []
    
    print("\n=== VENUE MATCHING LOGS ===")
    for nf in note_files:
        arxiv_id = nf.stem
        content = nf.read_text(encoding="utf-8", errors="ignore")
        
        fm_match = re.match(r"^---\s*\n(.*?)\n---\s*\n?(.*)", content, re.DOTALL)
        if not fm_match:
            continue
        try:
            fm = yaml.safe_load(fm_match.group(1)) or {}
        except Exception:
            continue
            
        pred_title = fm.get("title", "")
        pred_year = fm.get("year")
        pred_venue = fm.get("venue", "")
        pred_authors = fm.get("authors", [])
        if isinstance(pred_authors, str):
            pred_authors = [pred_authors]
            
        if arxiv_id not in gold_standard:
            unmatched_papers.append({
                "arxiv_id": arxiv_id,
                "reason": "No Semantic Scholar record fetched successfully"
            })
            continue
            
        gold = gold_standard[arxiv_id]
        
        title_ok = int(normalise(pred_title) == normalise(gold["title"]))
        year_ok = int(pred_year == gold["year"])
        venue_ok = int(fuzzy_venue_match(pred_venue, gold["venue"]))
        author_metrics = compute_set_prf(pred_authors, gold["authors"])
        
        print(f"  [{arxiv_id}] Pred Venue: '{pred_venue}' | Gold Venue: '{gold['venue']}' -> Match: {bool(venue_ok)}")
        
        group_a_details.append({
            "arxiv_id": arxiv_id,
            "title_match": title_ok,
            "year_match": year_ok,
            "venue_match": venue_ok,
            "authors_prf": author_metrics
        })
    print("===========================\n")
    
    n_matched = len(group_a_details)
    if n_matched == 0:
        return {"error": "No papers matched for evaluation"}
        
    agg_title = sum(r["title_match"] for r in group_a_details) / n_matched
    agg_year = sum(r["year_match"] for r in group_a_details) / n_matched
    agg_venue = sum(r["venue_match"] for r in group_a_details) / n_matched
    
    agg_auth_p = sum(r["authors_prf"]["precision"] for r in group_a_details) / n_matched
    agg_auth_r = sum(r["authors_prf"]["recall"] for r in group_a_details) / n_matched
    agg_auth_f1 = sum(r["authors_prf"]["f1"] for r in group_a_details) / n_matched
    
    return {
        "corpus_size": len(note_files),
        "matched_count": n_matched,
        "unmatched_list": unmatched_papers,
        "title_exact_match": round(agg_title, 4),
        "year_exact_match": round(agg_year, 4),
        "venue_fuzzy_match": round(agg_venue, 4),
        "authors_precision": round(agg_auth_p, 4),
        "authors_recall": round(agg_auth_r, 4),
        "authors_f1": round(agg_auth_f1, 4),
        "per_paper_results": group_a_details
    }

In [24]:
# ===========================================================================
# 5. Group B — Semantic Alignment (Emile Vault & OpenAlex Check)
# ===========================================================================

def evaluate_group_b(notes_dir: Path, gold_metadata_path: Path, vault_arxiv_path: Path, output_openalex_path: Path) -> dict:
    # Load Emile's vault mappings
    with open(gold_metadata_path, encoding="utf-8") as f:
        gold_metadata = json.load(f)
    with open(vault_arxiv_path, encoding="utf-8") as f:
        gold_arxiv_map = json.load(f)
        
    arxiv_to_gold_title = {arxiv_id: gold_title for gold_title, arxiv_id in gold_arxiv_map.items()}
    overlap_ids = sorted(list(arxiv_to_gold_title.keys()))
    
    # Incremental Cache Loading for OpenAlex
    openalex_cache = {}
    if output_openalex_path.exists():
        with open(output_openalex_path, encoding="utf-8") as f:
            try:
                openalex_cache = json.load(f) or {}
            except Exception:
                openalex_cache = {}
                
    missing_oa_ids = [aid for aid in overlap_ids if aid not in openalex_cache]
    
    if missing_oa_ids:
        print(f"Group B: Found {len(openalex_cache)} cached OpenAlex records. Fetching {len(missing_oa_ids)} missing records...")
        cache_updated = False
        for idx, aid in enumerate(missing_oa_ids, 1):
            print(f"  [{idx}/{len(missing_oa_ids)}] Querying OpenAlex for {aid}...")
            record = fetch_openalex_record(aid)
            if record:
                oa_topics = [t.get("display_name") for t in record.get("topics", []) if t.get("display_name")]
                oa_subfields = [t.get("subfield", {}).get("display_name") for t in record.get("topics", []) if t.get("subfield", {}).get("display_name")]
                oa_keywords = [k.get("display_name") for k in record.get("keywords", []) if k.get("display_name")]
                
                openalex_cache[aid] = list(set(oa_topics + oa_subfields + oa_keywords))
                cache_updated = True
            else:
                openalex_cache[aid] = []
            time.sleep(1.0)
            
        if cache_updated:
            output_openalex_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_openalex_path, "w", encoding="utf-8") as f:
                json.dump(openalex_cache, f, indent=2, ensure_ascii=False)
    else:
        print(f"Group B: OpenAlex cache is complete. Loaded {len(openalex_cache)} records.")
        
    total_overlap = 0
    excluded_papers = []
    b1_results = []
    b2_results = []
    
    # Paper-title tags to exclude (bibliographic cross-references, not concept labels)
    PAPER_TITLE_TAGS = {
        "Flow Matching for Generative Modeling",
        "Flow Matching_ Matching flows instead of scores",
        "Mamba_ Linear-Time Sequence Modeling with Selective State Spaces",
        "Sigmoid loss for language image pre-training",
        "Learning transferable visual models from natural language supervision",
        "Bridging Machine Learning and Logical Reasoning by Abductive Learning",
    }
    
    for nf in notes_dir.glob("*.md"):
        arxiv_id = nf.stem
        if arxiv_id not in arxiv_to_gold_title:
            continue
            
        total_overlap += 1
        gold_title = arxiv_to_gold_title[arxiv_id]
        gold_m = gold_metadata.get(gold_title)
        
        raw_gold_topics = gold_m.get("topics", []) if gold_m else []
        # Clean gold topics: remove bibliographic paper-title tags
        emile_semantic = [t for t in raw_gold_topics if t not in PAPER_TITLE_TAGS]
        
        if not emile_semantic:
            excluded_papers.append({
                "arxiv_id": arxiv_id,
                "title": gold_title,
                "reason": "No usable semantic annotations (topics/subsets) found in Emile's note after removing titles"
            })
            continue
            
        content = nf.read_text(encoding="utf-8", errors="ignore")
        fm_match = re.match(r"^---\s*\n(.*?)\n---\s*\n?(.*)", content, re.DOTALL)
        if not fm_match:
            continue
        try:
            fm = yaml.safe_load(fm_match.group(1)) or {}
        except Exception:
            continue
            
        agent_topics = fm.get("hasTopic", []) or []
        if isinstance(agent_topics, str):
            agent_topics = [agent_topics]
        
        # B1 Strict uses hasTopic only, paper titles removed
        b1_prf = compute_set_prf(agent_topics, emile_semantic)
        b1_results.append({
            "arxiv_id": arxiv_id,
            "title": gold_title,
            "metrics": b1_prf
        })
        
        # B2: Supplementary OpenAlex Support Rate Check (still using hasTopic only for alignment consistency)
        agent_set_norm = {normalise(x) for x in agent_topics if normalise(x)}
        emile_set_norm = {normalise(x) for x in emile_semantic if normalise(x)}
        
        oa_topics = openalex_cache.get(arxiv_id, [])
        oa_set_norm = {normalise(x) for x in oa_topics if normalise(x)}
        
        agent_additional = agent_set_norm - emile_set_norm
        
        matched = len(agent_set_norm & emile_set_norm)
        supported = len(agent_additional & oa_set_norm)
        unsupported = len(agent_additional - oa_set_norm)
        
        support_rate = (supported / len(agent_additional)) if agent_additional else 1.0
        
        b2_results.append({
            "arxiv_id": arxiv_id,
            "title": gold_title,
            "support_rate": round(support_rate, 4),
            "labels_breakdown": {
                "matched_with_emile": matched,
                "not_emile_but_supported_by_openalex": supported,
                "unsupported": unsupported
            }
        })
        
    n_eval = len(b1_results)
    if n_eval == 0:
        return {"error": "No papers with usable semantic gold annotations found"}
        
    # B1 Aggregations
    b1_p = sum(r["metrics"]["precision"] for r in b1_results) / n_eval
    b1_r = sum(r["metrics"]["recall"] for r in b1_results) / n_eval
    b1_f = sum(r["metrics"]["f1"] for r in b1_results) / n_eval
    
    # B2 Aggregations
    b2_support_rate = sum(r["support_rate"] for r in b2_results) / n_eval
    
    avg_matched = sum(r["labels_breakdown"]["matched_with_emile"] for r in b2_results) / n_eval
    avg_supported = sum(r["labels_breakdown"]["not_emile_but_supported_by_openalex"] for r in b2_results) / n_eval
    avg_unsupported = sum(r["labels_breakdown"]["unsupported"] for r in b2_results) / n_eval
    
    return {
        "total_overlap_papers": total_overlap,
        "evaluated_count": n_eval,
        "excluded_list": excluded_papers,
        "B1_strict": {
            "precision": round(b1_p, 4),
            "recall": round(b1_r, 4),
            "f1": round(b1_f, 4),
            "per_paper": b1_results
        },
        "B2_supplementary": {
            "average_openalex_support_rate": round(b2_support_rate, 4),
            "averages_breakdown": {
                "matched_with_emile": round(avg_matched, 2),
                "not_emile_but_supported_by_openalex": round(avg_supported, 2),
                "unsupported": round(avg_unsupported, 2)
            },
            "per_paper": b2_results
        }
    }


In [25]:
# ===========================================================================
# 6. Execution Wrapper
# ===========================================================================

baseline_gold_out = Path("data/results/metadata_api_gold.json")
openalex_gold_out = Path("data/results/openalex_cache.json")
results_json_out = Path("data/results/rq2_results.json")

print("Running Group A Bibliographic Evaluation...")
group_a_scores = evaluate_group_a(notes_dir, baseline_gold_out)

print("\nRunning Group B Semantic Evaluation...")
group_b_scores = evaluate_group_b(notes_dir, gold_metadata_path, vault_arxiv_path, openalex_gold_out)

# Export final results to json
output_summary = {
    "normalization_rules": "WikiLinks stripped using canonical target, lowercase, remove punctuation, collapse whitespace",
    "openalex_record_matching": "Queried via filter=doi:https://doi.org/10.48550/arXiv.{arxiv_id}",
    "Group_A_Bibliographic": group_a_scores,
    "Group_B_Semantic": group_b_scores
}

results_json_out.parent.mkdir(parents=True, exist_ok=True)
with open(results_json_out, "w", encoding="utf-8") as f:
    json.dump(output_summary, f, indent=2, ensure_ascii=False)
print(f"\nEvaluation complete. Results saved -> {results_json_out}")

Running Group A Bibliographic Evaluation...
Group A: Cache is complete. Loaded 40 records.

=== VENUE MATCHING LOGS ===
  [1711.11157] Pred Venue: 'ICML 2018' | Gold Venue: 'International Conference on Machine Learning' -> Match: True
  [2005.11401] Pred Venue: 'NeurIPS 2020' | Gold Venue: 'Neural Information Processing Systems' -> Match: True
  [2007.01282] Pred Venue: '' | Gold Venue: 'Conference of the European Chapter of the Association for Computational Linguistics' -> Match: False
  [2012.13635] Pred Venue: 'arXiv' | Gold Venue: 'Artificial Intelligence' -> Match: False
  [2210.06726] Pred Venue: 'arXiv' | Gold Venue: 'arXiv.org' -> Match: True
  [2212.12393] Pred Venue: 'NeurIPS 2023' | Gold Venue: 'Neural Information Processing Systems' -> Match: True
  [2305.10250] Pred Venue: 'arXiv' | Gold Venue: 'AAAI Conference on Artificial Intelligence' -> Match: False
  [2305.19951] Pred Venue: 'NeurIPS 2023' | Gold Venue: 'Neural Information Processing Systems' -> Match: True
  [2306.0

In [26]:
# ===========================================================================
# 7. Results Display
# ===========================================================================

print("=" * 60)
print("=== FINAL RQ2 EVALUATION METRICS SUMMARY ===")
print("=" * 60)
print(f"Group A: Bibliographic Catalog Matching (n={group_a_scores.get('matched_count')} matched)")
print(f"  Title Exact Match:      {group_a_scores.get('title_exact_match'):.4f}")
print(f"  Year Exact Match:       {group_a_scores.get('year_exact_match'):.4f}")
print(f"  Venue Fuzzy Match:      {group_a_scores.get('venue_fuzzy_match'):.4f}")
print(f"  Authors F1-Score:       {group_a_scores.get('authors_f1'):.4f}")
print("-" * 60)
print(f"Group B1: Local Semantic Alignment (Strict Expert Vault, n={group_b_scores.get('evaluated_count')} evaluated)")
print("  Note: hasTopic only, paper-title gold labels removed")
print(f"  Precision:              {group_b_scores.get('B1_strict', {}).get('precision'):.4f}")
print(f"  Recall:                 {group_b_scores.get('B1_strict', {}).get('recall'):.4f}")
print(f"  F1-Score:               {group_b_scores.get('B1_strict', {}).get('f1'):.4f}")
print("=" * 60)
print(f"Excluded papers in Group B: {len(group_b_scores.get('excluded_list', []))}")


=== FINAL RQ2 EVALUATION METRICS SUMMARY ===
Group A: Bibliographic Catalog Matching (n=40 matched)
  Title Exact Match:      0.9750
  Year Exact Match:       0.7750
  Venue Fuzzy Match:      0.5000
  Authors F1-Score:       0.8250
------------------------------------------------------------
Group B1: Local Semantic Alignment (Strict Expert Vault, n=22 evaluated)
  Note: hasTopic only, paper-title gold labels removed
  Precision:              0.0762
  Recall:                 0.1508
  F1-Score:               0.0957
Excluded papers in Group B: 0
